### **Import libraries**: 

In [12]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import sys
import fastparquet
import warnings
from IPython.display import clear_output

### **Load Parquet file into a Pandas DataFrame object and calculate mean sentiment, sentiment standard deviation, number of articles, number of strongly negative articles, and sentiment momentum.**

In [13]:
def load_and_calculate(parquet_file_name_param):
    df_ratings = pd.read_parquet(parquet_file_name_param)
    tickers = list(set(df_ratings.index.get_level_values(0)))
    tickers.sort()

    df_alt_features = pd.DataFrame()

    tickers = ["AMD", "AMZN"]

    for ticker in tickers:
        df_dates = pd.to_datetime(df_ratings.loc[ticker].index.get_level_values(0))
        df_dates = list(set([date.date().strftime("%Y-%m-%d") for date in df_dates]))
        df_dates.sort(reverse=True)
        for date in df_dates:
            df_temp = df_ratings.loc[ticker].xs(date, level = 0, drop_level = False)
            #print(df_ratings.loc[ticker])
            #print(df_ratings.loc[ticker].xs(date, level = 0, drop_level = False)[["pos", "neg"]])
            sent_diff = df_temp["pos"].sub(df_temp["neg"])
            sent_mean = sent_diff.values.mean()
            sent_std = sent_diff.std()
            news_count = len(sent_diff)
            pct_strong_negative = round((len(df_temp[df_temp["neg"] > 0.7]) / len(df_temp))*100,2)
            try:
                sent_5_day_avg = np.mean([np.mean(df_ratings.loc[ticker].xs((pd.Timestamp(date) - timedelta(days = i)).strftime("%Y-%m-%d"), level = 0, drop_level = False)["pos"].sub(df_ratings.loc[ticker].xs((pd.Timestamp(date) - timedelta(days = i)).strftime("%Y-%m-%d"), level = 0, drop_level = False)["neg"])) for i in range(5)])
                sent_momentum = sent_mean - sent_5_day_avg
            except:
                sent_5_day_avg = np.nan
                sent_momentum = np.nan

            #print(pd.DataFrame(data = {"sent_mean" : [sent_mean], "sent_std" : [sent_std], 
                                                     # "news_count": [news_count], "pct_strong_negative": [pct_strong_negative], 
                                                     # "sent_momentum": [sent_momentum]}))

            
            
            df_alt_features = pd.concat([df_alt_features, pd.DataFrame(data = {"Ticker": ticker,
                                                                               "Date": date,
                                                                               "sent_mean" : [sent_mean], 
                                                                               "sent_std" : [sent_std], 
                                                                               "news_count": [news_count], 
                                                                               "pct_strong_negative": [pct_strong_negative], 
                                                                               "sent_momentum": [sent_momentum]})])

            

        







    df_alt_features = df_alt_features.reset_index(drop=True)
    df_alt_features["Date"] = pd.to_datetime(df_alt_features["Date"])
    df_alt_features.set_index(["Ticker", "Date"], inplace = True)
    df_alt_features.sort_index(inplace = True, level = 0, sort_remaining = False)


    print(df_alt_features)
    #df_alt_features.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)
        

    df_alt_features.to_parquet("5_year_ticker_headlines_with_finbert_rating_calculate(test).parquet")
    sys.exit()
            #print(sent_5_day_avg)
            #print(sent_mean)

            

            
        
        
    
    
    #print(tickers)

In [14]:
def main():
    load_and_calculate("5_year_ticker_headlines_with_finbert_rating.parquet")

main()

                   sent_mean  sent_std  news_count  pct_strong_negative  \
Ticker Date                                                               
AMD    2026-04-23   0.344365  0.384843          25                 0.00   
       2026-04-22   0.130985  0.530675          25                12.00   
       2026-04-21   0.140959  0.556127          25                12.00   
       2026-04-20   0.405122  0.561846          25                 8.00   
       2026-04-19   0.450355  0.410446          16                 0.00   
...                      ...       ...         ...                  ...   
AMZN   2021-04-28   0.073384  0.518942           9                11.11   
       2021-04-27   0.147884  0.263658           3                 0.00   
       2021-04-26   0.167592  0.347568           7                 0.00   
       2021-04-25   0.303326  0.185643           3                 0.00   
       2021-04-23   0.246033       NaN           1                 0.00   

                   sent_

SystemExit: 

/home/leventefo/Programming/git/algorithmic-trading/notebooks/data_team/phase3/env/lib64/python3.14/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
